# ⛰ Core Lab 11 — Batch Size and the Speed-up Ladder
**A Billion Small Steps · Hardware for AI · Step 5 · Book sections 5.4–5.5 · Foothold lessons 5.4–5.5**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

**What you should see:** throughput levelling off with batch size; a measured total speed-up at the same accuracy.

| Part | What you do | Book |
|---|---|---|
| A | CPU vs GPU training, and a batch-size sweep | 5.4 |
| B | The speed-up ladder on Fashion-MNIST | 5.5 |

⏱️ About 2 hours. 📘 Textbook: D2L Ch 12–13 · Hwu, Kirk & El Hajj Ch 16. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
from torchvision import datasets                 # ready-made datasets (MNIST, Fashion-MNIST) for Part B

The same MNIST data, wide network and helpers as Core Lab 10, Part B (only the definitions; no training yet):

In [ ]:
from torchvision import datasets   # ready-made datasets, including MNIST
mnist = datasets.MNIST("data", train=True, download=True)   # the 60,000 handwritten training digits
mnist_test = datasets.MNIST("data", train=False, download=True)   # and 10,000 test digits
Xtr = (mnist.data.float() / 255).view(-1, 784).to(device); Ytr = mnist.targets.to(device)   # all images as rows of 784 pixels (0..1), and labels, on the GPU
Xte = (mnist_test.data.float() / 255).view(-1, 784).to(device); Yte = mnist_test.targets.to(device)   # the same for the test set

def wide_net():   # a wide network (2,048 neurons per layer), so the GPU has plenty of work
    torch.manual_seed(0)   # the same starting weights every time: a fair comparison
    return nn.Sequential(nn.Linear(784, 2048), nn.ReLU(), nn.Linear(2048, 2048), nn.ReLU(), nn.Linear(2048, 10)).to(device)   # 784 → 2048 → 2048 → 10, on the GPU

def accuracy(model, X, Y):   # the fraction of images whose highest score is the right digit
    with torch.no_grad():   # no training here, so keep no history
        return (model(X).argmax(1) == Y).float().mean().item()   # argmax(1) = the class with the highest score

def train_epoch(model, X, Y, batch=256, amp=False, lr=0.1):   # one epoch; amp=True turns on mixed precision
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)   # gradient descent with momentum
    scaler = torch.amp.GradScaler("cuda", enabled=amp)   # loss scaling (does nothing when amp is off)
    lossf = nn.CrossEntropyLoss()   # the loss for "which class?" problems
    for idx in torch.randperm(len(X), device=device).split(batch):   # shuffle the image numbers, cut them into batches
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):   # inside this block, big multiplies run in FP16
            loss = lossf(model(X[idx]), Y[idx])   # forward pass and loss for this batch
        opt.zero_grad(set_to_none=True)   # clear the old slopes
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()   # scale the loss up, backward, update, adjust the scale

## Part A · Batch Size and Training on the GPU  *(Section 5.4)*

**CPU vs GPU** for the wide network: time 100 training steps on each.

In [ ]:
def steps_time(dev, steps=100, batch=256):   # time training steps of the wide network on device dev
    torch.manual_seed(0)   # the same starting weights every time
    net = nn.Sequential(nn.Linear(784, 2048), nn.ReLU(), nn.Linear(2048, 2048), nn.ReLU(), nn.Linear(2048, 10)).to(dev)   # the wide network on dev
    opt = torch.optim.SGD(net.parameters(), lr=0.05); lossf = nn.CrossEntropyLoss()   # optimiser and loss
    X = torch.rand(batch, 784, device=dev); Y = torch.randint(0, 10, (batch,), device=dev)   # one fixed batch of made-up data
    def run():   # the steps themselves
        for _ in range(steps):   # one training step each time round
            loss = lossf(net(X), Y); opt.zero_grad(); loss.backward(); opt.step()   # forward, clear slopes, backward, update
    run()   # a warm-up run
    if dev == "cuda": torch.cuda.synchronize()   # wait for the GPU before starting the clock
    t0 = time.perf_counter(); run()   # the timed run
    if dev == "cuda": torch.cuda.synchronize()   # and wait again before stopping it
    return time.perf_counter() - t0   # seconds for all the steps

t_cpu = steps_time("cpu", steps=20) * 5   # 20 steps on the CPU, scaled up to 100 (the CPU is slow!)
t_gpu = steps_time(device)   # 100 steps on the GPU
print(f"CPU: {t_cpu:6.2f} s per 100 steps")   # show both...
print(f"GPU: {t_gpu:6.2f} s per 100 steps   → {t_cpu / t_gpu:.1f}× faster")   # ...and the speed-up

**Batch-size sweep:** throughput (images per second) and peak GPU memory.

In [ ]:
batches, tput, mem = [16, 64, 256, 1024, 4096], [], []   # batch sizes to try; results go in tput and mem
for B in batches:   # for each batch size
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()   # free cached memory and reset the "highest so far" counter
    # time 30 training steps, three times, and keep the best (a GPU's clock speed jumps about)
    t = min(steps_time(device, steps=30, batch=B) for _ in range(3))
    tput.append(30 * B / t)   # throughput = images processed ÷ seconds
    mem.append(torch.cuda.max_memory_allocated() / 1e9)   # the most GPU memory used, in GB
    print(f"batch {B:5d}: {tput[-1]:10,.0f} images/s   peak memory {mem[-1]:.2f} GB")   # show both

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))                 # two charts side by side: ax[0] and ax[1]
ax[0].semilogx(batches, tput, "o-"); ax[0].set_xlabel("batch size"); ax[0].set_ylabel("images / s")     # throughput
ax[0].set_title("Throughput rises, then levels off"); ax[0].grid(alpha=0.3)                            # its title and grid
ax[1].semilogx(batches, mem, "s-", c="tab:red"); ax[1].set_xlabel("batch size"); ax[1].set_ylabel("GB") # memory
ax[1].set_title("Memory keeps climbing"); ax[1].grid(alpha=0.3)                                        # its title and grid
plt.tight_layout(); plt.show()                                  # stop them overlapping, then draw

### ✏️ YOUR TURN (A)
1. How much faster was the GPU for the wide network? Compare with the small Core Lab 3 network (Section 5.4 explains the difference).
2. Where did throughput level off? Which batch size would you choose, and why?

*Your answers:*

## Part B · The Speed-Up Ladder  *(Section 5.5)*

**What is Fashion-MNIST?** A harder twin of MNIST, released by the online shop Zalando in 2017. Same format — 70,000 grey 28 × 28 pictures (60,000 training + 10,000 test), 10 classes — but the pictures are **clothing** instead of digits: T-shirt/top, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot. Telling a shirt from a pullover is harder than telling a 1 from a 7, so accuracy is lower (about 80–88% for a simple network instead of 97%).

Network 784 → 1024 → 1024 → 10. Each rung adds ONE change. We time one epoch and check test accuracy at every rung.

In [ ]:
from torchvision import transforms   # ways to convert images, e.g. into tensors
from torch.utils.data import DataLoader   # serves a dataset in batches
fash = datasets.FashionMNIST("data", train=True, download=True, transform=transforms.ToTensor())   # 60,000 clothing pictures, as tensors
fash_test = datasets.FashionMNIST("data", train=False, download=True)   # 10,000 test pictures
FXte = (fash_test.data.float() / 255).view(-1, 784).to(device); FYte = fash_test.targets.to(device)   # the test set as one tensor on the GPU
FX = (fash.data.float() / 255).view(-1, 784).to(device); FY = fash.targets.to(device)   # the training set too (used by rung 5)

# the six rungs of the ladder, one change each
RUNGS = ["0 baseline: batch 32, .item() every step", "1 + no per-step sync", "2 + num_workers=2, pin_memory, non_blocking",
         "3 + batch 256 (lr scaled up)", "4 + mixed precision", "5 + whole dataset on the GPU"]  # (the list goes on)

def ladder(rung):   # train one epoch with every improvement up to this rung switched on
    torch.manual_seed(0)   # the same starting weights on every rung
    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 1024), nn.ReLU(), nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 10)).to(device)   # 784 → 1024 → 1024 → 10
    B = 256 if rung >= 3 else 32   # rung 3 and up: a bigger batch
    opt = torch.optim.SGD(model.parameters(), lr=0.2 if rung >= 3 else 0.05, momentum=0.9)   # with a bigger learning rate
    amp = rung >= 4   # rung 4 and up: mixed precision
    scaler = torch.amp.GradScaler("cuda", enabled=amp); lossf = nn.CrossEntropyLoss()   # loss scaling and the loss
    if rung >= 5:   # rung 5: the data is already on the GPU
        batches = ((FX[i], FY[i]) for i in torch.randperm(len(FX), device=device).split(B))   # just shuffle and slice it: no DataLoader, no copies
    else:   # rungs 0–4 use a DataLoader
        workers = 2 if rung >= 2 else 0   # rung 2 and up: two helper processes...
        batches = DataLoader(fash, batch_size=B, shuffle=True, num_workers=workers, pin_memory=(rung >= 2))   # ...and pinned memory
    log, running = [], torch.zeros((), device=device)   # a Python list (rung 0) or a total kept on the GPU (rung 1 and up)
    sync(); t0 = time.perf_counter()   # start timing the epoch
    for x, y in batches:   # one batch at a time
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)   # copy it to the GPU (does nothing if already there)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):   # FP16 inside this block, if amp is on
            loss = lossf(model(x), y)   # forward pass and loss
        opt.zero_grad(set_to_none=True)   # clear the old slopes
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()   # backward and update
        if rung == 0:   # the baseline...
            log.append(loss.item())   # ...waits for the GPU every step
        else:   # rung 1 and up...
            running += loss.detach()   # ...keep the total on the GPU: no waiting
    sync(); seconds = time.perf_counter() - t0   # stop the clock
    return seconds, accuracy(model, FXte, FYte)   # the epoch's time, and the test accuracy

rung_res = []   # (seconds, accuracy) for each rung
for r, name in enumerate(RUNGS):   # climb the ladder, rung 0 to 5
    rung_res.append(ladder(r))   # train one epoch on this rung
    s, acc = rung_res[-1]   # its time and accuracy
    print(f"{name:46s} {s:6.2f} s   {rung_res[0][0] / s:5.1f}× faster   test accuracy {acc:.1%}")   # compare with rung 0

In [ ]:
names = ["T-shirt", "trouser", "pullover", "dress", "coat", "sandal", "shirt", "sneaker", "bag", "boot"]   # the 10 classes, in order
plt.figure(figsize=(10, 1.6))                                    # one wide, short drawing area
for i in range(10):                                              # the first 10 test pictures
    plt.subplot(1, 10, i + 1)                                    # 1 row of 10 boxes: draw in box i+1
    plt.imshow(fash_test.data[i], cmap="gray")                   # test picture i
    plt.title(names[fash_test.targets[i]], fontsize=8)           # its label, as a word
    plt.axis("off")                                              # no axes around the picture
plt.show()                                                       # draw them

Not every rung helps on every machine, and that is the real lesson. Rung 4 (mixed precision) may do little or even slow things down: while the CPU still loads and copies every batch, faster GPU arithmetic barely shows. After one epoch, a wobble of 2–4% in accuracy between rungs is normal noise from different batch orders and sizes: re-run a rung to check before believing it.

### ✏️ YOUR TURN (B) — record your results
| Rung | Seconds per epoch | Speed-up | Test accuracy |
|---|---|---|---|
| 0 baseline | | 1.0× | |
| 1 no sync | | | |
| 2 workers + pinned | | | |
| 3 batch 256 | | | |
| 4 mixed precision | | | |
| 5 data on GPU | | | |

1. Which single rung gave the biggest jump? Explain why with a unit you studied.
2. Did any rung change the accuracy noticeably?
3. Did you reach the 5× target?
4. Rung 5 keeps mixed precision switched on. In the `ladder` cell, change `amp = rung >= 4` to `amp = rung == 4`, re-run that cell (it climbs the whole ladder again, about half a minute), then run `ladder(5)` in a new cell. Is rung 5 faster without mixed precision? Then change `B = 256 if rung >= 3 else 32` to `B = 1024 if rung >= 3 else 32` and compare `ladder(5)` with and without mixed precision again. When does mixed precision pay off, and why? *(Our T4 numbers are in Section 5.5.)*

*Your answers:*

---
## 🎓 Wrap-up
- Bigger batches keep the GPU busy until throughput levels off; memory keeps growing.
- Climb the ladder one change at a time and measure each rung: not every rung helps on every machine.
- **Next — Core Lab 12: Will It Fit? Plan a Project.**